In [13]:
# Necessary libraries
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score
from transformers import ASTForAudioClassification
import wandb
from tqdm import tqdm


In [14]:
# Configurations
random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed_all(1234)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [15]:
CACHE_DIR  = "../working/ast_cache"
MODEL_DIR  = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

GENRES       = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

In [16]:
# phase 1: head-only warmup
PHASE1_EPOCHS = 5
PHASE1_LR     = 1e-3

# phase 2: partial unfreeze of last 2 transformer encoder blocks
PHASE2_EPOCHS    = 10
PHASE2_LR_HEAD   = 1e-4
PHASE2_LR_TRUNK  = 1e-5
N_UNFREEZE_BLOCKS = 2

BATCH_SIZE   = 32
WEIGHT_DECAY = 0.01
NUM_WORKERS  = 0

In [17]:
from torch.utils.data import Dataset
class PrecomputedASTDataset(Dataset):
    def __init__(self, cache_dir, split):
        self.split_dir = os.path.join(cache_dir, split)
        self.files     = sorted(os.listdir(self.split_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fpath = os.path.join(self.split_dir, self.files[idx])
        data  = torch.load(fpath, weights_only=True)
        return data["input_values"], data["label"]

In [18]:
train_dataset = PrecomputedASTDataset(CACHE_DIR, "train")
val_dataset   = PrecomputedASTDataset(CACHE_DIR, "val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

Train samples: 1600 | Val samples: 400


In [6]:
# AST Model
model = ASTForAudioClassification.from_pretrained(
    AST_MODEL_ID,
    num_labels=10,
    ignore_mismatched_sizes=True
)

# freeze entire backbone for phase 1
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

model = model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable parameters: {trainable:,}")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Phase 1 trainable parameters: 9,226


In [19]:
def run_epoch(model, loader, criterion, optimizer, training, device, desc=""):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for inputs, labels in tqdm(loader, desc=desc):
            inputs, labels = inputs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(inputs).logits
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss  += loss.item() * inputs.size(0)
            preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())

    n        = len(loader.dataset)
    avg_loss = total_loss / n
    f1       = f1_score(labels_all, preds, average="macro")
    acc      = accuracy_score(labels_all, preds)
    return avg_loss, f1, acc


criterion = nn.CrossEntropyLoss()

In [ ]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-src-phase1",
    config={
        "model":            AST_MODEL_ID,
        "phase1_epochs":    PHASE1_EPOCHS,
        "phase1_lr":        PHASE1_LR,
        "phase2_epochs":    PHASE2_EPOCHS,
        "phase2_lr_head":   PHASE2_LR_HEAD,
        "phase2_lr_trunk":  PHASE2_LR_TRUNK,
        "n_unfreeze_blocks": N_UNFREEZE_BLOCKS,
        "batch_size":       BATCH_SIZE,
        "weight_decay":     WEIGHT_DECAY,
        "augmentation":     "cross-song mixing, per-stem gain, variable SNR",
    }
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Atul\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [10]:
# phase 1: train classifier head only
best_val_f1 = 0.0
print("Phase 1: head warmup")
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

for epoch in range(PHASE1_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True, device=DEVICE
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False, device=DEVICE
    )
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1, "phase": 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss": val_loss,     "val_f1": val_f1,     "val_acc": val_acc,
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[P1] Epoch {epoch+1}/{PHASE1_EPOCHS}: "
          f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "ast_src_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Phase 1: head warmup
[P1] Epoch 1/5: train_f1=0.5372 | val_f1=0.6926
  Saved best model (val_f1=0.6926)
[P1] Epoch 2/5: train_f1=0.7605 | val_f1=0.7327
  Saved best model (val_f1=0.7327)
[P1] Epoch 3/5: train_f1=0.8138 | val_f1=0.7371
  Saved best model (val_f1=0.7371)
[P1] Epoch 4/5: train_f1=0.8463 | val_f1=0.7663
  Saved best model (val_f1=0.7663)
[P1] Epoch 5/5: train_f1=0.8652 | val_f1=0.7660


In [9]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-src-phase2-2blocks",
    config={
        "model":             AST_MODEL_ID,
        "phase2_epochs":     PHASE2_EPOCHS,
        "phase2_lr_head":    PHASE2_LR_HEAD,
        "phase2_lr_trunk":   PHASE2_LR_TRUNK,
        "n_unfreeze_blocks": N_UNFREEZE_BLOCKS,
        "batch_size":        BATCH_SIZE,
        "weight_decay":      WEIGHT_DECAY,
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Atul\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [8]:
model.load_state_dict(torch.load(
    os.path.join(MODEL_DIR, "ast_src_best.pth"),
    map_location=DEVICE, weights_only=True
))
print("Phase 1 weights reloaded.")

for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False
print("Phase 1 weights reloaded, backbone fully frozen.")

Phase 1 weights reloaded.
Phase 1 weights reloaded, backbone fully frozen.


In [10]:
# phase 2: unfreeze last N transformer encoder blocks
print(f"\nPhase 2: unfreezing last {N_UNFREEZE_BLOCKS} encoder blocks")

encoder_blocks = model.audio_spectrogram_transformer.encoder.layer
for block in encoder_blocks[-N_UNFREEZE_BLOCKS:]:
    for param in block.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 trainable parameters: {trainable:,}")

best_val_f1 = 0.0

# differential learning rates: lower LR for unfrozen trunk, higher for head
head_params  = list(model.classifier.parameters())
trunk_params = [p for block in encoder_blocks[-N_UNFREEZE_BLOCKS:]
                for p in block.parameters()]

optimizer = optim.AdamW([
    {"params": head_params,  "lr": PHASE2_LR_HEAD},
    {"params": trunk_params, "lr": PHASE2_LR_TRUNK},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

for epoch in range(PHASE2_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS} [Train]")
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS} [Val]")
    scheduler.step()

    wandb.log({
        "epoch": PHASE1_EPOCHS + epoch + 1, "phase": 2,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss": val_loss,     "val_f1": val_f1,     "val_acc": val_acc,
        "lr_head":  optimizer.param_groups[0]["lr"],
        "lr_trunk": optimizer.param_groups[1]["lr"],
    })
    print(f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS}: "
          f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "ast_src_phase2_best.pth.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")


Phase 2: unfreezing last 2 encoder blocks
Phase 2 trainable parameters: 14,184,970


[P2] Epoch 1/10 [Val]: 100%|██████████| 13/13 [01:20<00:00,  6.16s/it]


[P2] Epoch 1/10: train_f1=0.8589 | val_f1=0.7798
  Saved best model (val_f1=0.7798)


[P2] Epoch 2/10 [Val]: 100%|██████████| 13/13 [01:19<00:00,  6.14s/it]


[P2] Epoch 2/10: train_f1=0.8948 | val_f1=0.7815
  Saved best model (val_f1=0.7815)


[P2] Epoch 3/10 [Val]: 100%|██████████| 13/13 [01:20<00:00,  6.17s/it]


... (11 outputs left out)

[P2] Epoch 9/10 [Val]: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]


[P2] Epoch 9/10: train_f1=0.9812 | val_f1=0.8022


[P2] Epoch 10/10 [Val]: 100%|██████████| 13/13 [01:19<00:00,  6.10s/it]

[P2] Epoch 10/10: train_f1=0.9837 | val_f1=0.8022


In [11]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
lr_head,█▇▇▆▅▃▂▂▁▁
lr_trunk,█▇▇▆▅▃▂▂▁▁
phase,▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▆▇▇███
train_f1,▁▃▄▅▆▇▇███
train_loss,█▆▅▄▃▂▂▁▁▁
val_acc,▁▂▄▅▅▆█▇██
val_f1,▁▂▄▆▅▆█▇██
val_loss,█▆▄▂▂▂▁▁▁▁
epoch,15


In [20]:
# Reload Phase 2 model checkpoint, unfreeze last 3 blocks

model = ASTForAudioClassification.from_pretrained(
    AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
)
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

model.load_state_dict(torch.load(
    os.path.join(MODEL_DIR, "ast_src_phase2_best.pth.pth"),
    map_location=DEVICE, weights_only=True
))
model = model.to(DEVICE)
print("Phase 2 weights loaded, backbone fully frozen.")

encoder_blocks = model.audio_spectrogram_transformer.encoder.layer
for block in encoder_blocks[-3:]:
    for param in block.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 3 trainable parameters: {trainable:,}")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Phase 2 weights loaded, backbone fully frozen.
Phase 3 trainable parameters: 21,272,842


In [21]:
PHASE3_EPOCHS   = 5
PHASE3_LR_HEAD  = 5e-5
PHASE3_LR_TRUNK = 5e-6

head_params  = list(model.classifier.parameters())
trunk_params = [p for block in encoder_blocks[-3:]
                for p in block.parameters()]

optimizer = optim.AdamW([
    {"params": head_params,  "lr": PHASE3_LR_HEAD},
    {"params": trunk_params, "lr": PHASE3_LR_TRUNK},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE3_EPOCHS)

In [22]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-src-phase3-3blocks",
    config={
        "model":             AST_MODEL_ID,
        "phase3_epochs":     PHASE3_EPOCHS,
        "phase3_lr_head":    PHASE3_LR_HEAD,
        "phase3_lr_trunk":   PHASE3_LR_TRUNK,
        "n_unfreeze_blocks": 3,
        "batch_size":        BATCH_SIZE,
        "starting_from":     "ast_src_phase2_best.pth.pth",
    }
)

In [23]:
best_val_f1_p3 = 0.8026

for epoch in range(PHASE3_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True,
        device=DEVICE, desc=f"[P3] Epoch {epoch+1}/{PHASE3_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False,
        device=DEVICE, desc=f"[P3] Epoch {epoch+1}/{PHASE3_EPOCHS} [Val]"
    )
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr_head":  optimizer.param_groups[0]["lr"],
        "lr_trunk": optimizer.param_groups[1]["lr"],
    })
    print(f"[P3] Epoch {epoch+1}/{PHASE3_EPOCHS}: "
          f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1_p3:
        best_val_f1_p3 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "ast_src_phase3_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

wandb.finish()
print(f"Training complete. Best phase 3 val F1: {best_val_f1_p3:.4f}")

[P3] Epoch 1/5 [Val]: 100%|██████████| 13/13 [01:16<00:00,  5.92s/it]


[P3] Epoch 1/5: train_f1=0.9731 | val_f1=0.7883


[P3] Epoch 2/5 [Val]: 100%|██████████| 13/13 [21:32<00:00, 99.40s/it] 


[P3] Epoch 2/5: train_f1=0.9843 | val_f1=0.8011


[P3] Epoch 3/5 [Val]: 100%|██████████| 13/13 [21:35<00:00, 99.62s/it] 


[P3] Epoch 3/5: train_f1=0.9925 | val_f1=0.7951


... (2 outputs left out)

[P3] Epoch 5/5 [Val]: 100%|██████████| 13/13 [21:38<00:00, 99.86s/it] 


[P3] Epoch 5/5: train_f1=0.9969 | val_f1=0.8042
  Saved best model (val_f1=0.8042)


epoch,▁▃▅▆█
lr_head,█▆▄▂▁
lr_trunk,█▆▄▂▁
train_acc,▁▄▇▇█
train_f1,▁▄▇▇█
train_loss,█▅▃▂▁
val_acc,▁▇▃▇█
val_f1,▁▇▄▇█
val_loss,█▇▂▁▁
epoch,5
lr_head,0


Training complete. Best phase 3 val F1: 0.8042
